# GRACE reproduced paper-based — contract v2

Đối chứng theo contract cho reveal. Đọc README và `RESEARCH.md` trước khi so sánh. Đây là tái lập có các thích nghi đã công khai; không phải khôi phục bit-for-bit run tác giả.

Notebook dùng Python/WSL hiện có, Joern và CodeT5 local. Các stage ghi checkpoint/manifest riêng; hai profile dùng chung cache đã xác minh. Cấu hình hiện tại chạy full test, prepare, freeze và inference Vertex AI với tối đa 10 request đồng thời; Joern dùng 4 worker.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "GRACE-original-code").is_dir() and (p / "Graduation-Project").is_dir())
sys.path.insert(0, str(ROOT / "Graduation-Project/Ablation_V2/Reproduced-Code-Based"))
import importlib
import reproduction_pipeline
reproduction_pipeline = importlib.reload(reproduction_pipeline)
Config, Pipeline = reproduction_pipeline.Config, reproduction_pipeline.Pipeline

DATASET = 'reveal'
SPLIT = "test"  # full frozen test cohort
MODEL_DIR = str(ROOT / (
    "Graduation-Project/Ablation_V2/Reproduced-Code-Based/"
    "kaggle_gpu_job/downloads/"
    "quangdung1111-grace-kaggle-gpu-ipynb/downloads/model"
))
BATCH_SIZE = 4  # lower peak RAM for CodeT5
WORKERS = 4  # up to four Joern processes at a time
INFERENCE_WORKERS = 10  # concurrent Vertex AI requests; independent of Joern
JOERN_SHARD_INDEX = int(os.environ.get("JOERN_SHARD_INDEX", "0"))
JOERN_SHARD_COUNT = int(os.environ.get("JOERN_SHARD_COUNT", "1"))
MAX_TARGETS = None  # full train and full frozen test cohort
RUN_PREPARE = True
FREEZE = True
RUN_INFERENCE = True
if JOERN_SHARD_COUNT > 1:
    FREEZE = False
    RUN_INFERENCE = False  # shards only build Joern graph caches
config = Config(variant='paper', dataset=DATASET, split=SPLIT,
                root=ROOT, model_dir=MODEL_DIR, max_targets=MAX_TARGETS, batch_size=BATCH_SIZE,
                workers=WORKERS, inference_workers=INFERENCE_WORKERS,
                joern_shard_index=JOERN_SHARD_INDEX, joern_shard_count=JOERN_SHARD_COUNT,
                reuse_legacy_reveal_test_embeddings=True)
pipeline = Pipeline(config)
print("Python:", sys.executable)
print("Joern:", os.environ.get("JOERN_HOME"))
print("Output:", pipeline.output)


Python: /usr/bin/python3
Joern: /home/yungn/tools/joern/y/joern-cli
Output: /mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reproduced-Paper-Based/reveal/test


## Cách chạy

Chọn **kernel venv WSL hiện có → Restart Kernel → Run All**. Notebook chạy trực tiếp trong venv được chọn; cell cấu hình in đường dẫn Python và Joern để đối chiếu. Cấu hình `.env` được nạp tự động. Không cần khởi động batch để chạy notebook thủ công. Để chia Joern qua nhiều máy, đặt cùng JOERN_SHARD_COUNT=N và mỗi máy dùng JOERN_SHARD_INDEX khác nhau từ 0 đến N-1. Mỗi máy cần checkout/cache riêng. Run All khi shard count > 1 chỉ tạo cache shard, tự tắt freeze và inference; gom các joern_shard_*.zip về một máy, merge bằng stage merge-joern-shards rồi chạy prepare đầy đủ không shard.

Cache/checkpoint hợp lệ được giữ và tái sử dụng khi Run All lại; `RUN_INFERENCE=True` gọi Vertex thật sau prepare/freeze với tối đa 10 request đồng thời. Joern giữ tối đa 4 worker.

## 1. Audit data và khả năng tái sử dụng

Xác minh SHA-256 parquet, cohort test đã freeze, ID/code hash, full train và source code cạnh CPG binary. Pool loại mọi bản sao code mục tiêu. ReVeal validation được tạo bằng adapter canonical, Devign validation giữ cohort của Full.

In [2]:
audit = pipeline.audit()
print(json.dumps(audit, ensure_ascii=False, indent=2))


{
  "schema": "grace_reproduction_v1",
  "variant": "paper",
  "dataset": "reveal",
  "split": "test",
  "contract_sha256": "becd38a9afd9677f2cdee47b1669f0442e53dd82c4b001c137d90d83a6710b3c",
  "dataset_manifest_sha256": "c1f40056953fa3062867e0e9dfdf1dda2a1f7fcfd9c37ffab9a0f0cba1fc31cc",
  "target_count": 2274,
  "train_count": 18187,
  "target_identity": "b52d9c9a20cd77c4b62b1cd6a7436f6a557f6791e5fea3d3f1d3690042de4d32",
  "train_identity": "c4c3e80a79615dc7c2bd4195d953cc821632d3642fa66f4be1926d5ea0a36035",
  "train_target_exact_code_overlap": 56,
  "same_code_candidates_excluded": true,
  "graph_reuse": {
    "graph_index": "/mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reveal/Code-CPG/reveal_test_full/graphs.jsonl",
    "graph_index_sha256": "09fdf2241b0f7e0a6e2a899f327c2f5d078001158cc0529c9fb64b705959e863",
    "target_count": 2274,
    "approved_target_records": 2274,
    "approved_unique_code_binaries": 2272,
    "rejection_reasons": {},
    "recovered_missing_paths

## 2. Chuẩn bị retrieval, whole CPG và prompt

Đặt `RUN_PREPARE=True` ở cell cấu hình rồi chạy lại. Pipeline dùng cùng initial top-50 với Full, chọn semantic top-5, rerank lexical/syntactic rồi lấy một demo train có nhãn. Profile code: whitening/IP và sequence ratio trên AST edge strings; profile paper: t-SNE/L2 và Joern AST SimSBT. CodeT5 checkpoint và các lựa chọn paper chưa quy định được ghi vào manifest.

CPG binary chỉ tái sử dụng khi source hash khớp; xuất whole GraphML/AST/CFG/PDG vào cache mới. CPG thiếu mới chạy Joern parse. Riêng ReVeal test bật tái sử dụng vector CodeT5 cũ sau khi kiểm tra code hash toàn cohort theo đúng thứ tự; model/tokenizer revision chưa xác minh và được ghi rõ là giả định. Train vẫn encode bằng checkpoint hiện tại. Devign không dùng cache positional cũ. GraphVerb đã slice không được cắm vào baseline. Lần đầu có thể tốn thời gian cho embedding và graph của các demo; lần sau dùng cache có fingerprint.

In [3]:
if RUN_PREPARE:
    prepared = pipeline.prepare()
    print("Prepared:", prepared["status"])
else:
    print("Audit only. Set RUN_PREPARE=True to prepare all stages.")


Joern features: 6590 unique code records
Joern features: 25/6590
Joern features: 50/6590
Joern features: 75/6590
Joern features: 100/6590
Joern features: 125/6590
Joern features: 150/6590
Joern features: 175/6590
Joern features: 200/6590
Joern features: 225/6590
Joern features: 250/6590
Joern features: 275/6590
Joern features: 300/6590
Joern features: 325/6590
Joern features: 350/6590
Joern features: 375/6590
Joern features: 400/6590
Joern features: 425/6590
Joern features: 450/6590
Joern features: 475/6590
Joern features: 500/6590
Joern features: 525/6590
Joern features: 550/6590
Joern features: 575/6590
Joern features: 600/6590
Joern features: 625/6590
Joern features: 650/6590
Joern features: 675/6590
Joern features: 700/6590
Joern features: 725/6590
Joern features: 750/6590
Joern features: 775/6590
Joern features: 800/6590
Joern features: 825/6590
Joern features: 850/6590
Joern features: 875/6590
Joern features: 900/6590
Joern features: 925/6590
Joern features: 950/6590
Joern featur

## 3. Freeze cấu hình và prompt

Sau khi xem prompt/manifest trên validation, chốt cấu hình. Nếu chuyển sang test, chạy audit và prepare trên toàn bộ test rồi freeze artefact đó. Freeze kiểm tra drift ở inference; nó không biến test đã dùng trước đây thành test mới.

In [4]:
if FREEZE:
    frozen = pipeline.freeze()
    print("Frozen:", pipeline.output / "freeze.json")


Frozen: /mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reproduced-Paper-Based/reveal/test/freeze.json


## 4. Inference và evaluation chung

Chạy sau prepare + freeze. Thiết lập `VERTEX_PROJECT` và `VERTEX_LOCATION` trong môi trường trước khi mở notebook. `INFERENCE_WORKERS=10` giới hạn số request Vertex đồng thời; mỗi worker tái sử dụng client riêng, một luồng ghi checkpoint khi từng request hoàn thành. Model, temperature, seed, max tokens và retry lấy trực tiếp từ contract. Checkpoint chỉ tái sử dụng nếu prompt và cấu hình inference trùng. File kết quả cuối giữ thứ tự target; label mục tiêu chỉ dùng tại evaluator.

In [5]:
if RUN_INFERENCE:
    metrics = pipeline.infer()
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("No API requests. RUN_INFERENCE=False.")


inference: 0/2274 cached; 10 Vertex workers
inference: 25/2274
inference: 50/2274
inference: 75/2274
inference: 100/2274
inference: 125/2274
inference: 150/2274
inference: 175/2274
inference: 200/2274
inference: 225/2274
inference: 250/2274
inference: 275/2274
inference: 300/2274
inference: 325/2274
inference: 350/2274
inference: 375/2274
inference: 400/2274
inference: 425/2274
inference: 450/2274
inference: 475/2274
inference: 500/2274
inference: 525/2274
inference: 550/2274
inference: 575/2274
inference: 600/2274
inference: 625/2274
inference: 650/2274
inference: 675/2274
inference: 700/2274
inference: 725/2274
inference: 750/2274
inference: 775/2274
inference: 800/2274
inference: 825/2274
inference: 850/2274
inference: 875/2274
inference: 900/2274
inference: 925/2274
inference: 950/2274
inference: 975/2274
inference: 1000/2274
inference: 1025/2274
inference: 1050/2274
inference: 1075/2274
inference: 1100/2274
inference: 1125/2274
inference: 1150/2274
inference: 1175/2274
inference: 

## Cách diễn giải

So sánh Full với bản tái lập này cho tác động tổng hợp của phương pháp cải tiến trong điều kiện contract. Leave-one-out hiện tại trả lời đóng góp từng factor trong Full. Hai phép so sánh có mục đích khác nhau. Tắt mọi No-C hiện có không tự khôi phục GRACE: graph gốc phải còn, retrieval gốc phải được thay lại, và metric contract vẫn giữ.